In [4]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import warnings
import os

warnings.filterwarnings('ignore', category=UserWarning)

print("--- Step 1: Connecting Cloud Streams via KaggleHub ---")
try:
    import kagglehub
    downloaded_folder_path = kagglehub.dataset_download("austinreese/craigslist-carstrucks-data")
    file_path = os.path.join(downloaded_folder_path, "vehicles.csv")
    print(f"File verified on cloud server at: {file_path}")
except Exception as e:
    print(f"KaggleHub connection hit a hitch: {e}")
    file_path = None

# Expanded list of target columns requested by the user
target_columns = ['price', 'year', 'odometer', 'lat', 'model', 'size', 'type', 'manufacturer', 'cylinders', 'state', 'condition']
categorical_features = ['model', 'size', 'type', 'manufacturer', 'cylinders', 'state', 'condition']
routing_features = ['year', 'odometer', 'lat']
all_features = routing_features + categorical_features

if file_path and os.path.exists(file_path):
    print("\nStreaming massive dataset... Extracting requested continuous and categorical dimensions...")
    # Stream in ONLY the specific columns needed to preserve system memory
    df_raw = pd.read_csv(file_path, usecols=target_columns)
    print(f"Data stream successful! Loaded {len(df_raw):,} raw listings.")
else:
    print("\n[Fallback] Generating mock marketplace array containing requested categorical traits...")
    n_samples = 50000
    df_raw = pd.DataFrame({
        'price': np.random.uniform(2000, 45000, n_samples),
        'year': np.random.randint(2005, 2026, n_samples),
        'odometer': np.random.uniform(5000, 200000, n_samples),
        'lat': np.random.uniform(24.0, 49.0, n_samples),
        'manufacturer': np.random.choice(['toyota', 'ford', 'chevrolet', 'honda', 'nissan'], n_samples),
        'model': np.random.choice(['camry', 'f-150', 'silverado', 'civic', 'altima'], n_samples),
        'condition': np.random.choice(['excellent', 'good', 'fair', 'like new'], n_samples),
        'cylinders': np.random.choice(['4 cylinders', '6 cylinders', '8 cylinders'], n_samples),
        'size': np.random.choice(['compact', 'mid-size', 'full-size'], n_samples),
        'type': np.random.choice(['sedan', 'truck', 'SUV', 'coupe'], n_samples),
        'state': np.random.choice(['ca', 'ny', 'tx', 'fl', 'il'], n_samples)
    })

# --- Step 2: Outlier Scrubbing & Text Missing Value Imputation ---
print("\n--- Step 2: Scrubbing Noise and Handling Missing Text Fields ---")
# Basic price and mileage sanity check boundaries
df_sanitized = df_raw[
    (df_raw['price'] > 1000) & (df_raw['price'] < 150000) &
    (df_raw['year'] > 2000) & (df_raw['year'] <= 2026) &
    (df_raw['odometer'] > 100) & (df_raw['odometer'] < 400000)
].copy()

# Fill missing categorical strings with 'unknown' so the encoder doesn't crash
for col in categorical_features:
    df_sanitized[col] = df_sanitized[col].fillna('unknown').astype(str).str.lower()

# Fill missing numerical fields with median averages
num_imputer = SimpleImputer(strategy='median')
df_sanitized[routing_features] = num_imputer.fit_transform(df_sanitized[routing_features])

df_sanitized = df_sanitized.reset_index(drop=True)
print(f"Sanitization complete. Retained {len(df_sanitized):,} structured listings.")

# --- Step 3: Transforming Text Categories into Numerical Matrix ---
print("\n--- Step 3: Ordinal Encoding Categorical Text Columns to Int Vectors ---")
# Converting raw string labels into clean distinct numerical identifiers
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
df_sanitized[categorical_features] = encoder.fit_transform(df_sanitized[categorical_features])

X_core = df_sanitized[all_features]
y_target = df_sanitized['price']

# --- Step 4: Split Into Training and Evaluation Sets ---
X_train, X_test, y_train, y_test = train_test_split(X_core, y_target, test_size=0.2, random_state=42)

# --- Step 5: Slicing 3D Spatial Cloud into Neighborhoods ---
print("\n--- Step 4: Computing Spatial Neighborhood Partitions via 3D Coordinates ---")
kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
# We partition the market topology based purely on Year, Mileage, and Latitude spatial coordinates
train_clusters = kmeans.fit_predict(X_train[routing_features])

# --- Step 6: Distributed Expert Forest Training ---
print("\n--- Step 5: Distributing Upgraded High-Dimensional Vectors to Experts ---")
neighborhood_models = {}

for cluster_id in range(10):
    chunk_mask = (train_clusters == cluster_id)
    X_chunk = X_train[chunk_mask]
    y_chunk = y_train[chunk_mask]

    # Experts now evaluate the full vector including brand, condition, and configuration traits
    expert_model = RandomForestRegressor(n_estimators=40, max_depth=15, random_state=42, n_jobs=-1)
    expert_model.fit(X_chunk, y_chunk)
    neighborhood_models[cluster_id] = expert_model
    print(f" > High-D Expert Model #{cluster_id} successfully finalized on {len(X_chunk):,} entries.")

# --- Step 7: Final Performance Validation Metric Evaluation ---
print("\n--- Step 6: Running System Diagnostic Performance Evaluation ---")
test_clusters = kmeans.predict(X_test[routing_features])
test_predictions = []

# Validate performance by matching names correctly
for i, (idx, car_sample) in enumerate(X_test.iterrows()):
    assigned_cluster = test_clusters[i]
    single_car_df = pd.DataFrame([car_sample], columns=all_features)
    pred = neighborhood_models[assigned_cluster].predict(single_car_df)[0]
    test_predictions.append(pred)

mae = mean_absolute_error(y_test, test_predictions)
r2 = r2_score(y_test, test_predictions)

print(f"\n>> Upgraded Mean Absolute Error (MAE): ${mae:,.2f}")
print(f">> Upgraded R² (Market Variation Explained): {r2 * 100:.2f}%")

# --- Step 8: Custom Inference Tool ---
print("\n--- Step 7: Live Custom Market Query Tester ---")
def price_custom_car(year, odometer, lat, manufacturer, model, condition, cylinders, size, type, state):
    # 1. Construct input data matching the structural names
    raw_input = pd.DataFrame([{
        'year': year, 'odometer': odometer, 'lat': lat,
        'manufacturer': str(manufacturer).lower(), 'model': str(model).lower(),
        'condition': str(condition).lower(), 'cylinders': str(cylinders).lower(),
        'size': str(size).lower(), 'type': str(type).lower(), 'state': str(state).lower()
    }])

    # 2. Match the encoding values built during the training phase
    raw_input[categorical_features] = encoder.transform(raw_input[categorical_features])

    # 3. CRITICAL FIX: Force the columns to perfectly match the exact order used during model.fit()
    raw_input = raw_input[all_features]

    # 4. Route based on the core 3D variables
    cluster_routing = kmeans.predict(raw_input[routing_features])[0]
    predicted_val = neighborhood_models[cluster_routing].predict(raw_input)[0]

    print(f"Target routed to Expert Branch: #{cluster_routing}")
    print(f"Calibrated Fair Value Guess: ${predicted_val:,.2f}")

# Example Test Call: Running the exact test instance again
price_custom_car(
    year=2015, odometer=75000, lat=37.7,
    manufacturer='toyota', model='camry', condition='excellent',
    cylinders='4 cylinders', size='mid-size', type='sedan', state='ca'
)

--- Step 1: Connecting Cloud Streams via KaggleHub ---
Using Colab cache for faster access to the 'craigslist-carstrucks-data' dataset.
File verified on cloud server at: /kaggle/input/craigslist-carstrucks-data/vehicles.csv

Streaming massive dataset... Extracting requested continuous and categorical dimensions...
Data stream successful! Loaded 426,880 raw listings.

--- Step 2: Scrubbing Noise and Handling Missing Text Fields ---
Sanitization complete. Retained 345,090 structured listings.

--- Step 3: Ordinal Encoding Categorical Text Columns to Int Vectors ---

--- Step 4: Computing Spatial Neighborhood Partitions via 3D Coordinates ---

--- Step 5: Distributing Upgraded High-Dimensional Vectors to Experts ---
 > High-D Expert Model #0 successfully finalized on 36,076 entries.
 > High-D Expert Model #1 successfully finalized on 42,824 entries.
 > High-D Expert Model #2 successfully finalized on 17,727 entries.
 > High-D Expert Model #3 successfully finalized on 31,668 entries.
 > Hi

In [ ]:
import joblib

print("--- Step 10: Packaging and Exporting Your AI Architecture ---")

# 1. Bundle all system components into a single Python dictionary
production_package = {
    'kmeans': kmeans,
    'encoder': encoder,
    'neighborhood_models': neighborhood_models,
    'routing_features': routing_features,
    'categorical_features': categorical_features,
    'all_features': all_features
}

# 2. Serialize the bundle into a compressed binary file
model_filename = 'craigslist_engine.joblib'
joblib.dump(production_package, model_filename, compress=3)
print(f"[Success] System compiled and saved locally as '{model_filename}'")

--- Step 10: Packaging and Exporting Your AI Architecture ---
[Success] System compiled and saved locally as 'craigslist_engine.joblib'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>